####Objetivo del notebook

Revisión general de las 41 variables categóricas que comprenden la primera base de datos derivada*, que contiene víctimas previas de VIF y VP que no presentan un registro asociado de lesión fatal por causa externa en el marco de tiempo 2015-2026, las cuales son nuestro primer grupo de interés (esta base proviene del contraste por número de cédula entre la base de lesiones no fatales de causa externa  2015-2024** y la base lesiones fatales de causa externa 2015- junio 2026***, luego se dejan solo las filas no coincidentes de la base de lesiones no fatales de causa externa, para luego filtar por el campo "Contexto del Hecho" los casos VIF contra niños, ñiñas y adolescentes, VIF contra el adulto mayor, VIF entre otros familiares y VP), la cual esta compuesta por 675401 casos, para:

a. Generar gráficas por variable, que permitan comparar la distribución de estas para el primer grupo de interés, con las de la base de datos de casos de VIF-VP previa y que posteriormente mueren por violencia de esta naturaleza (segundo grupo de interés, base: df_victimas_VIF_VP_con_muerte_homicidio en el notebook_01_4), para establecer por observación cuales variables presentan diferencias entre grupos y luego aplicar el estadístico de prueba.

b. Establecer que variables de estudio requieren que las categorías sean reagrupadas. 

-Entrada:

Base derivada: vif_vp_en_fatales_no_coincidentes

-Salida:

c. Guardar en fomato delta  la base de datos de nuestro primer grupo de interés que denominaremos "no fatales" con nombre VIF_VP_no_fatales_v1.

*La base de datos derivada fue entregada depurada por Instituto Nacional de Medicina Legal y Ciencias Forenses de Colombia (INMLCF) segun solicitud. Debido a que el número de cédula es un dato anonimizado. 

**Lesiones no fatales por causa externa: es una lesión ocasionada por una causa externa en la que la víctima sobrevive y requiere valoración médico-legal.

***Lesiones fatales de causa externa: es una lesión ocasionada por una causa externa que produce la muerte de la víctima.




In [0]:
#Importar librerías
from pyspark.sql.functions import col, sum, count, when, create_map, lit, lower
import matplotlib.pyplot as plt
from itertools import chain

In [0]:
#Leer tabla de datos como un data frame
 
df_VIF_VP_no_fatales = spark.table("ml_proyecto_7405607705157039.default.vif_vp_en_fatales_no_coincidentes")

In [0]:
#Visualización del data frame

display(df_VIF_VP_no_fatales)

In [0]:
#Numero de registros

df_VIF_VP_no_fatales.count()

In [0]:
#verificar si hay nulos en todo el dataframe

hay_nulos = df_VIF_VP_no_fatales.filter(
    " OR ".join([f"`{c}` IS NULL" for c in df_VIF_VP_no_fatales.columns])
).count()

print(hay_nulos)

In [0]:
#Descripción de columnas

df_VIF_VP_no_fatales.printSchema()

In [0]:
#Renombrar columnas

df_VIF_VP_no_fatales = (
df_VIF_VP_no_fatales.withColumnRenamed('ID', 'id') 
.withColumnRenamed('Año del hecho', 'año_del_hecho')
.withColumnRenamed('Sexo de la Víctima', 'sexo_victima')
.withColumnRenamed('Grupo de edad quinquenal', 'grupo_edad_quinquenal')
.withColumnRenamed('Grupo Mayor Menor de Edad', 'grupo_mayor_menor_edad')
.withColumnRenamed('Grupo de Edad judicial', 'grupo_edad_judicial')
.withColumnRenamed('Ciclo Vital', 'ciclo_vital')
.withColumnRenamed('País de Nacimiento', 'pais_de_nacimiento')
.withColumnRenamed('Escolaridad', 'escolaridad')
.withColumnRenamed('Estado Civil', 'estado_civil')
.withColumnRenamed('Tipo de Discapacidad', 'tipo_discapacidad')
.withColumnRenamed('Pertenencia Étnica', 'pertenencia_etnica')
.withColumnRenamed('Orientación Sexual', 'orientacion_sexual')
.withColumnRenamed('Identidad de Género', 'identidad_de_genero')
.withColumnRenamed('Transgénero', 'transgenero')
.withColumnRenamed('Pertenencia Grupal', 'pertenencia_grupal')
.withColumnRenamed('Mes del hecho', 'mes_del_hecho')
.withColumnRenamed('Día del Hecho', 'dia_del_hecho')
.withColumnRenamed('Rango de Hora del Hecho (X3 Horas)', 'rango_hora_hecho_por3h')
.withColumnRenamed('Codigo Dane Municipio', 'codigo_dane_municipio')
.withColumnRenamed('Municipio del hecho DANE', 'municipio_del_hecho_dane')
.withColumnRenamed('Departamento del hecho DANE', 'departamento_del_hecho_dane')
.withColumnRenamed('Codigo Dane Departamento', 'codigo_dane_departamento')
.withColumnRenamed('Localidad del Hecho', 'localidad_del_hecho')
.withColumnRenamed('Zona del Hecho', 'zona_del_hecho')
.withColumnRenamed('Escenario del Hecho', 'escenario_del_hecho')
.withColumnRenamed('Actividad Durante el Hecho', 'actividad_durante_hecho')
.withColumnRenamed('Circunstancia del Hecho Detallada', 'circunstancia_del_hecho_detallada')
.withColumnRenamed('Contexto del Hecho', 'contexto_del_hecho')
.withColumnRenamed('Mecanismo Causal de la Lesión no Fatal', 'mecanismo_causal')
.withColumnRenamed('Diagnóstico Topográfico de la Lesión no Fatal', 'diagnostico_topografico')
.withColumnRenamed('Sexo del Agresor', 'sexo_del_agresor')
.withColumnRenamed('Presunto Agresor Detallado', 'presunto_agresor')
.withColumnRenamed('Factor Desencadenante de la Agresión', 'factor_desencadenante_agresion')
.withColumnRenamed('Condición de la Víctima (AT)', 'condicion_victima_AT')
.withColumnRenamed('Medio de Desplazamiento o Transporte', 'medio_Desplazamiento')
.withColumnRenamed('Servicio del Vehículo', 'servicio_vehiculo')
.withColumnRenamed('Clase o Tipo de Accidente de Transporte', 'tipo_accidente_transporte')
.withColumnRenamed('Objeto de Colisión', 'objeto_de_colision')
.withColumnRenamed('Servicio del Objeto de Colisión', 'servicio_objeto_colision')
.withColumnRenamed('Días de Incapacidad Medicolegal', 'dias_de_incapacidad_medicolegal')
.withColumnRenamed('Pueblo Indígena', 'pueblo_indigena')
)


In [0]:
#Descripción de columnas renombradas

df_VIF_VP_no_fatales.printSchema()

In [0]:
df_VIF_VP_no_fatales.select("año_del_hecho") \
    .distinct() \
    .orderBy("año_del_hecho") \
    .show()



In [0]:
#Gráfica de barras año del hecho

#Rango de tiempo en el que se dieron los hechos, se encuentra establecido por el período que transcurre desde el día 1 de #enero hasta el 31 de diciembre.

# Agrupar y ordenar por año
df_año_del_hecho = (
    df_VIF_VP_no_fatales
    .groupBy("año_del_hecho")
    .agg(count("*").alias("cantidad"))
    .orderBy(col("año_del_hecho"))
)

# Convertir a pandas
pdf = df_año_del_hecho.toPandas()

plt.figure(figsize=(10,5))

plt.bar(pdf["año_del_hecho"], pdf["cantidad"])

plt.title("Distribución de víctimas previas de VIF o VP y estan vivas por año")
plt.xlabel("Año")
plt.ylabel("Cantidad")

# Mostrar todos los años de 2015 a 2025
plt.xticks(range(2015, 2025))

plt.grid(axis='y', alpha=0.3)

plt.show()

In [0]:
#Gráfica circular del sexo de la victima

#Hace referencia al sexo del lesionado. Desde la perspectiva biológica, se refiere a las características genéticas, endocrinas #y morfológicas del cuerpo. Las categorías utilizadas para clasificar estas características en los seres humanos son hombre, #mujer.

#agrupar por sexo
df_sexo_victima = (
    df_VIF_VP_no_fatales
    .groupBy("sexo_victima")
    .agg(count("*").alias("cantidad"))
)

#convertir en pandas
pdf = df_sexo_victima.toPandas()

#crear gráfica cirular

#función para mostrar cantidad y porcentaje
def formato(pct):
    total = pdf["cantidad"].sum()
    cantidad = int(round(pct * total / 100))
    return f'{cantidad}\n({pct:.1f}%)'

plt.figure(figsize=(7,7))

plt.pie(
    pdf["cantidad"],
    labels=pdf["sexo_victima"],
    autopct=formato,
    startangle=90
)

plt.title("Distribución de víctimas previas de VIF o VP y estan vivas por sexo")

plt.show()

In [0]:
#Gráfica de Barras con Grupo de edad quinquenal (grupos etáreos “rangos quinquenales”) 

df_grupo_edad_quinquenal = (
    df_VIF_VP_no_fatales
    .groupBy("grupo_edad_quinquenal")
    .agg(count("*").alias("cantidad"))
    .orderBy(col("grupo_edad_quinquenal"))
)

# Convertir a pandas
pdf = df_grupo_edad_quinquenal.toPandas()

plt.figure(figsize=(10,5))

plt.bar(pdf["grupo_edad_quinquenal"], pdf["cantidad"])

plt.title("Número de casos por grupo etáreo quinquenal")
plt.xlabel("Año")
plt.ylabel("Cantidad")

plt.xticks(rotation=90)
plt.grid(axis='y', alpha=0.3)

plt.show()

In [0]:
#Valores unicos de la variable grupo_mayor_menor_edad
df_VIF_VP_no_fatales.select("grupo_mayor_menor_edad").distinct().show(truncate=False)

In [0]:
# Gráfica de barras grupo_mayor_menor_edad
	
#Agrupación según se trate de personas mayores o menores de edad (mayoría de edad a partir de los 18 años, Constitución Política de Colombia 1991).

#Recodificar valores de las variable grupo_mayor_menor_edad
df_VIF_VP_en_fatales_no_coincidentes = df_VIF_VP_no_fatales.withColumn(
    "grupo_mayor_menor_edad",
     when(col("grupo_mayor_menor_edad") == "a) Menor de Edad (<18 Años)", "<18 años")
    .when(col("grupo_mayor_menor_edad") == "a) Menores de Edad (<18 años)", "<18 años")
    .when(col("grupo_mayor_menor_edad") == "b) Mayores de Edad (>18 años)", ">=18 años")
    .when(col("grupo_mayor_menor_edad") == "b) Mayor de Edad (>18 Años)", ">=18 años")
    .otherwise(col("grupo_mayor_menor_edad"))
)
    
#Agrupar por categoría de edad, contar casos y ordenar
df_grupo_mayor_menor_edad = (
    df_VIF_VP_no_fatales
    .groupBy("grupo_mayor_menor_edad")
    .agg(count("*").alias("cantidad"))
    .orderBy(col("grupo_mayor_menor_edad"))    
)

# Convertir a pandas
pdf = df_grupo_mayor_menor_edad.toPandas()

plt.figure(figsize=(10,5))

plt.bar(pdf["grupo_mayor_menor_edad"], pdf["cantidad"])

plt.title("Número de casos por dos grupos etáreos mayores y menores de edad")
plt.xlabel("Año")
plt.ylabel("Cantidad")

plt.xticks(rotation=90)
plt.grid(axis='y', alpha=0.3)

plt.show()

In [0]:
#Valores unicos de la variable grupo_edad_judicial 
df_VIF_VP_no_fatales.select("grupo_edad_judicial").distinct().show(truncate=False)

In [0]:
#Gráfica de barras Grupo de Edad judicial

#Agrupación de casos en rangos de edades que son de interés judicial.

#Agrupar por categoría, contar casos y ordenar
df_grupo_edad_judicial = (
    df_VIF_VP_no_fatales
    .groupBy("grupo_edad_judicial")
    .agg(count("*").alias("cantidad"))
    .orderBy(col("grupo_edad_judicial"))
)

# Convertir a pandas
pdf = df_grupo_edad_judicial.toPandas()

plt.figure(figsize=(10,5))

plt.bar(pdf["grupo_edad_judicial"], pdf["cantidad"])

plt.title("Número de casos por grupo de edad judicial")
plt.xlabel("Año")
plt.ylabel("Cantidad")

plt.xticks(rotation=90)
plt.grid(axis='y', alpha=0.3)

plt.show()




In [0]:
#Valores unicos de la variable ciclo_vital 
df_VIF_VP_no_fatales.select("ciclo_vital").distinct().show(truncate=False)

In [0]:
# Gráfica de barras ciclo_vital

#Corresponde al establecimiento de la etapa de desarrollo vital en que se encontraba la víctima, en el momento de realizar el examen medicolegal.

#Agrupar por categoría, contar casos y ordenar
df_ciclo_vital = (
    df_VIF_VP_no_fatales
    .groupBy("ciclo_vital")
    .agg(count("*").alias("cantidad"))
    .orderBy(col("ciclo_vital"))
)

# Convertir a pandas
pdf = df_ciclo_vital.toPandas()

plt.figure(figsize=(10,5))

plt.bar(pdf["ciclo_vital"], pdf["cantidad"])

plt.title("Número de casos por grupo de ciclo vital")
plt.xlabel("Ciclo vital")
plt.ylabel("Cantidad")

plt.xticks(rotation=90)
plt.grid(axis='y', alpha=0.3)

plt.show()

In [0]:
df_VIF_VP_no_fatales.select("pais_de_nacimiento") \
    .distinct() \
    .orderBy("pais_de_nacimiento") \
    .show(100, truncate=False)

In [0]:
#Gráfica de barras pais_de_nacimiento

#Lugar de nacimiento registrado en el documento de identidad.

#Agrupar por categoría, contar casos y ordenar
df_pais_de_nacimiento = (
    df_VIF_VP_no_fatales
    .groupBy("pais_de_nacimiento")
    .agg(count("*").alias("cantidad"))
    .orderBy(col("pais_de_nacimiento"))
    .limit(20)
)

# Convertir a pandas
pdf = df_pais_de_nacimiento.toPandas()

plt.figure(figsize=(10,5))

plt.bar(pdf["pais_de_nacimiento"], pdf["cantidad"])

plt.title("Número de casos por país de nacimiento")
plt.xlabel("País de nacimiento")
plt.ylabel("Cantidad")

plt.xticks(fontsize=8)
plt.xticks(rotation=90)
plt.grid(axis='y', alpha=0.3)

plt.show()

In [0]:
#Valores unicos de la variable escolaridad
df_VIF_VP_no_fatales.select("escolaridad").distinct().show(truncate=False)

In [0]:
#Gráfica de barras de escolaridad

#Se refiere al grado de escolaridad más alto al cual ha llegado la persona de acuerdo con los niveles del sistema educativo #formal: preescolar, básica en sus niveles de primaria, secundaria, media y superior.


#Recodificar valores de las variable
df_VIF_VP_no_fatales = df_VIF_VP_no_fatales.withColumn(
    "escolaridad",
     when(col("escolaridad") == "Educación básica secundaria o secundaria baja", "Secundaria")
    .when(col("escolaridad") == "Educación media o secundaria alta", "Secundaria")
    .when(col("escolaridad") == "Educación básica primaria", "Primaria")
    .when(col("escolaridad") == "Educación inicial y educación preescolar", "Preescolar")
    .when(col("escolaridad") == "Maestría", "Superior")
    .when(col("escolaridad") == "Universitario", "Superior")
    .when(col("escolaridad") == "Especialización, Maestría o equivalente", "Superior")
    .when(col("escolaridad") == "Básica secundaria", "Secundaria")
    .when(col("escolaridad") == "MAESTRÍA", "Superior")
    .when(col("escolaridad") == "Doctorado o equivalente", "Superior")
    .when(col("escolaridad") == "Tecnológica", "Superior")
    .when(col("escolaridad") == "Profesional", "Superior")
    .when(col("escolaridad") == "Básica primaria", "Primaria")
    .when(col("escolaridad") == "Educación técnica profesional y tecnológica", "Superior")
    .when(col("escolaridad") == "Especialización, maestría o equivalente", "Superior")    
    .otherwise(col("escolaridad"))
)

#Agrupar por categoría, contar casos y ordenar
df_escolaridad = (
    df_VIF_VP_no_fatales
    .groupBy("escolaridad")
    .agg(count("*").alias("cantidad"))
    .orderBy(col("escolaridad"))
)

# Convertir a pandas
pdf = df_escolaridad.toPandas()

plt.figure(figsize=(10,5))

plt.bar(pdf["escolaridad"], pdf["cantidad"])

plt.title("Número de casos por nivel de escolaridad de la victima")
plt.xlabel("Escolaridad")
plt.ylabel("Cantidad")

plt.xticks(rotation=90)
plt.grid(axis='y', alpha=0.3)

plt.show()

In [0]:
#Valores unicos de la variable estado_civil
df_VIF_VP_no_fatales.select("estado_civil").distinct().show(truncate=False)

In [0]:
#Gráfica de barras estado_civil

#Es la situación de cada persona en relación con las leyes o costumbres relativas al matrimonio que existen en el país.

#Recodificar valores de las variable
df_VIF_VP_no_fatales = df_VIF_VP_no_fatales.withColumn(
    "estado_civil",
     when(col("estado_civil") == "Separado (a), divorciado (a)", "Separado (a), Divorciado (a)")      
    .otherwise(col("estado_civil"))
)

#Agrupar por categoría, contar casos y ordenar
df_estado_civil = (
    df_VIF_VP_no_fatales
    .groupBy("estado_civil")
    .agg(count("*").alias("cantidad"))
    .orderBy(col("cantidad").desc())
)

# Convertir a pandas
pdf = df_estado_civil.toPandas()

plt.figure(figsize=(10,5))

plt.bar(pdf["estado_civil"], pdf["cantidad"])

plt.title("Número de casos por estado civil de la victima")
plt.xlabel("Estado civil")
plt.ylabel("Cantidad")

plt.xticks(rotation=90)
plt.grid(axis='y', alpha=0.3)

plt.show()

In [0]:
#Gráfica de barras tipo_discapacidad

#De acuerdo a la variable, se establece en la valoración médico legal, si la víctima presenta algún tipo de discapacidad (física, auditiva, mental, visual, síquica, intelectual, psicosocial y múltiple).

#Agrupar por categoría, contar casos y ordenar
df_tipo_discapaciad = (
    df_VIF_VP_no_fatales
    .groupBy("tipo_discapacidad")
    .agg(count("*").alias("cantidad"))
    .orderBy(col("tipo_discapacidad"))
)

# Convertir a pandas
pdf = df_tipo_discapaciad.toPandas()

plt.figure(figsize=(10,5))

plt.bar(pdf["tipo_discapacidad"], pdf["cantidad"])

plt.title("Número de casos por tipo de discapacidad de la victima")
plt.xlabel("Tipo de discapacidad")
plt.ylabel("Cantidad")

plt.xticks(rotation=90)
plt.grid(axis='y', alpha=0.3)

plt.show()

In [0]:
#Valores unicos de la variable pertinencia_etnica
df_VIF_VP_no_fatales.select("pertenencia_etnica").distinct().show(truncate=False)

In [0]:
# Gráfica de barras pertinencia_etnica

#Se refiere a la identificación de las personas como integrantes de alguno de los cuatro grupos étnicos reconocidos en #Colombia (población Afrocolombiana o Afrodescendiente; Negra, Raizal del Archipiélago de San Andrés, Providencia y Santa #Catalina; o Palenquera de San Basilio de Palenque - Mathes - Bolívar); Indígena y Gitano - Rrom; se registra la información #al momento de la valoración medicolegal.

#Recodificar valores de las variable pertenencia_etnica
df_VIF_VP_no_fatales = df_VIF_VP_no_fatales.withColumn(
    "pertenencia_etnica",
     when(col("pertenencia_etnica") == "ROM (Gitano)", "Rom (Gitano)")
    .when(col("pertenencia_etnica") == "Sin Pertenencia Étnica", "Sin pertenencia étnica")         
    .otherwise(col("pertenencia_etnica"))
)

#Agrupar por categoría, contar casos y ordenar
df_pertenencia_etnica = (
    df_VIF_VP_no_fatales
    .groupBy("pertenencia_etnica")
    .agg(count("*").alias("cantidad"))
    .orderBy(col("pertenencia_etnica"))
)

# Convertir a pandas
pdf = df_pertenencia_etnica.toPandas()

plt.figure(figsize=(10,5))

plt.bar(pdf["pertenencia_etnica"], pdf["cantidad"])

plt.title("Número de casos por tipo de pertenencia étnica de la victima")
plt.xlabel("Pertenencia étnica")
plt.ylabel("Cantidad")

plt.xticks(rotation=90)
plt.grid(axis='y', alpha=0.3)

plt.show()

In [0]:
# Gráfica de barras orientacion_sexual

#Se refiere a la atracción afectiva, erótica o sexual de una persona hacia otras, según el sexo de las personas hacia las que #se orienta el deseo; las categorías son heterosexual, homosexual, bisexual, asexual, no sabe / no informa. En lesionados se #recopila la información de acuerdo a lo referido por la víctima en la valoración medicolegal.

#Agrupar por categoría, contar casos y ordenar
df_orientacion_sexual = (
    df_VIF_VP_no_fatales
    .groupBy("orientacion_sexual")
    .agg(count("*").alias("cantidad"))
    .orderBy(col("orientacion_sexual"))
)

# Convertir a pandas
pdf = df_orientacion_sexual.toPandas()

plt.figure(figsize=(10,5))

plt.bar(pdf["orientacion_sexual"], pdf["cantidad"])

plt.title("Número de casos por orientación sexual de la victima")
plt.xlabel("Orientación sexual")
plt.ylabel("Cantidad")

plt.xticks(rotation=90)
plt.grid(axis='y', alpha=0.3)

plt.show()


In [0]:
# Gráfica de barras identidad_de_genero

#En lo que respecta a “Identidad de Género”, es el auto reconocimiento que una persona hace de sí misma a partir de la #construcción social, histórica y cultural de lo que se ha definido lo femenino, lo masculino o la transición entre ambos; las #categorías son: masculino, femenino y transgénero.

#Agrupar por categoría, contar casos y ordenar
df_identidad_de_genero = (
    df_VIF_VP_no_fatales
    .groupBy("identidad_de_genero")
    .agg(count("*").alias("cantidad"))
    .orderBy(col("identidad_de_genero"))
)

# Convertir a pandas
pdf = df_identidad_de_genero.toPandas()

plt.figure(figsize=(10,5))

plt.bar(pdf["identidad_de_genero"], pdf["cantidad"])

plt.title("Número de casos por identidad de género de la víctima")
plt.xlabel("Identidad de género")
plt.ylabel("Cantidad")

plt.xticks(rotation=90)
plt.grid(axis='y', alpha=0.3)

plt.show()

In [0]:
# Gráfica de barras transgenero

#Es la identidad en la que no coincide el sexo biológico con las características inscritas en lo que la expectativa colectiva #ha construido social, histórica y culturalmente como femenino o masculino.

#Recodificar valores de las variable grupo_mayor_menor_edad
df_VIF_VP_no_fatales = df_VIF_VP_no_fatales.withColumn(
    "transgenero",
     when(col("transgenero") == "No Aplica", "No aplica")   
    .otherwise(col("transgenero"))
)

#Agrupar por categoría, contar casos y ordenar
df_transgenero = (
    df_VIF_VP_no_fatales
    .groupBy("transgenero")
    .agg(count("*").alias("cantidad"))
    .orderBy(col("transgenero"))
)

# Convertir a pandas
pdf = df_transgenero.toPandas()

plt.figure(figsize=(10,5))

plt.bar(pdf["transgenero"], pdf["cantidad"])

plt.title("Número de casos por transgénero de la víctima")
plt.xlabel("Transgénero")
plt.ylabel("Cantidad")

plt.xticks(rotation=90)
plt.grid(axis='y', alpha=0.3)

plt.show()

In [0]:
#Valores unicos de la variable pertinencia_grupal
df_VIF_VP_no_fatales.select("pertenencia_grupal").distinct().show(truncate=False)

In [0]:
# Gráfica de barras pertenencia_grupal

#Se entienden como las características de las personas o grupos sociales, que los hacen más frágiles o susceptibles para #enfrentar los riesgos, lo que hace que la probabilidad de la ocurrencia de la violencia y de la afectación sea mayor que en #cualquier otra persona. Podrán constituir condición de vulnerabilidad, entre otras, las siguientes: la edad, la discapacidad, #la pertenencia a comunidades indígenas o a minorías, la victimización, la migración y el desplazamiento interno, la pobreza, #el género y la privación de libertad. 

#Agrupar por categoría, contar casos y ordenar
df_pertenencia_grupal = (
    df_VIF_VP_no_fatales
    .groupBy("pertenencia_grupal")
    .agg(count("*").alias("cantidad"))
    .orderBy(col("pertenencia_grupal"))
)

# Convertir a pandas
pdf = df_pertenencia_grupal.toPandas()

plt.figure(figsize=(10,5))

plt.bar(pdf["pertenencia_grupal"], pdf["cantidad"])

plt.title("Número de casos por pertenencia grupal de la víctima")
plt.xlabel("Pertenencia grupal")
plt.ylabel("Cantidad")

plt.xticks(rotation=90)
plt.grid(axis='y', alpha=0.3)

plt.show()


In [0]:
#Valores unicos de la variable mes_del_hecho
df_VIF_VP_no_fatales.select("mes_del_hecho").distinct().show(truncate=False)

In [0]:
# Gráfica de barras del mes_del_hecho

#Tiempo determinado por el mes en el cuál se establece el momento en que ocurrieron los hechos.

#Todos los meses en minusculas
df_VIF_VP_no_fatales = df_VIF_VP_no_fatales.withColumn(
    "mes_del_hecho",
    lower(col("mes_del_hecho"))
)

#Diccionario con el orden de los meses
orden_meses = {
    "enero": 1,
    "febrero": 2,
    "marzo": 3,
    "abril": 4,
    "mayo": 5,
    "junio": 6,
    "julio": 7,
    "agosto": 8,
    "septiembre": 9,
    "octubre": 10,
    "noviembre": 11,
    "diciembre": 12
}

#Convertir el diccionario a un mapa de Spark
mapa_meses = create_map(
    [lit(x) for x in chain(*orden_meses.items())]
)

#Agrupar por categoría, contar casos y ordenar
df_mes_del_hecho = (
    df_VIF_VP_no_fatales
    .withColumn("orden_mes", mapa_meses[col("mes_del_hecho")])
    .groupBy("mes_del_hecho", "orden_mes")
    .agg(count("*").alias("cantidad"))
    .orderBy("orden_mes")
)

# Convertir a pandas
pdf = df_mes_del_hecho.toPandas()

plt.figure(figsize=(10,5))

plt.bar(pdf["mes_del_hecho"], pdf["cantidad"])

plt.title("Número de casos por mes del hecho")
plt.xlabel("mes_del_hecho")
plt.ylabel("Cantidad")

plt.xticks(rotation=90)
plt.grid(axis='y', alpha=0.3)

plt.show()

In [0]:
#Valores unicos de la variable dia_del_hecho
df_VIF_VP_no_fatales.select("dia_del_hecho").distinct().show(truncate=False)

In [0]:
#Gráfica de barras dia_del_hecho

#Día de la semana en la que ocurrieron los hechos.

#Todos los días en minusculas
df_VIF_VP_no_fatales = df_VIF_VP_no_fatales.withColumn(
    "dia_del_hecho",
    lower(col("dia_del_hecho"))
)

#Diccionario con el orden de los días del hecho
orden_dias = {
    "lunes": 1,
    "martes": 2,
    "miércoles": 3,
    "jueves": 4,
    "viernes": 5,
    "sábado": 6,
    "domingo": 7,
}

#Convertir el diccionario a un mapa de Spark
mapa_dias = create_map(
    [lit(x) for x in chain(*orden_dias.items())]
)

#Agrupar por categoría, contar casos y ordenar
df_dia_del_hecho = (
    df_VIF_VP_no_fatales
    .withColumn("orden_dia", mapa_dias[col("dia_del_hecho")])
    .groupBy("dia_del_hecho", "orden_dia")
    .agg(count("*").alias("cantidad"))
    .orderBy("orden_dia")
)

# Convertir a pandas
pdf = df_dia_del_hecho.toPandas()

plt.figure(figsize=(10,5))

plt.bar(pdf["dia_del_hecho"], pdf["cantidad"])

plt.title("Número de casos por dia del hecho")
plt.xlabel("dia_del_hecho")
plt.ylabel("Cantidad")

plt.xticks(rotation=90)
plt.grid(axis='y', alpha=0.3)

plt.show()


In [0]:
#Gráfica de barras rango_hora_hecho_por3h

#Intervalo de tiempo o categoría cronológica, en el cual se establece que sucedieron los hechos.

#Diccionario con el orden de los rangos de hora
orden_horas = {
    "(00:00 a 02:59)": 1,
    "(03:00 a 05:59)": 2,
    "(06:00 a 08:59)": 3,
    "(09:00 a 11:59)": 4,
    "(12:00 a 14:59)": 5,
    "(15:00 a 17:59)": 6,
    "(18:00 a 20:59)": 7,
    "(21:00 a 23:59)": 8,
    "Sin información": 9,
    
}

#Convertir el diccionario a un mapa de Spark
mapa_horas = create_map(
    [lit(x) for x in chain(*orden_horas.items())]
)

#Agrupar por categoría, contar casos y ordenar
df_rango_hora_hecho_por3h = (
    df_VIF_VP_no_fatales
    .withColumn("orden_hora", mapa_horas[col("rango_hora_hecho_por3h")])
    .groupBy("rango_hora_hecho_por3h", "orden_hora")
    .agg(count("*").alias("cantidad"))
    .orderBy("orden_hora")
)

# Convertir a pandas
pdf = df_rango_hora_hecho_por3h.toPandas()

plt.figure(figsize=(10,5))

plt.bar(pdf["rango_hora_hecho_por3h"], pdf["cantidad"])

plt.title("Número de casos por rango de hora del hecho")
plt.xlabel("Rango de hora del hecho")
plt.ylabel("Cantidad")

plt.xticks(rotation=90)
plt.grid(axis='y', alpha=0.3)

plt.show()


In [0]:
# Gráfica de barras departamento_del_hecho_dane

#Agrupar por categoría, contar casos y ordenar
df_departamento_del_hecho_dane = (
    df_VIF_VP_no_fatales
    .groupBy("departamento_del_hecho_dane")
    .agg(count("*").alias("cantidad"))
    .orderBy(col("departamento_del_hecho_dane"))
    .orderBy(col("cantidad").desc())
)

# Convertir a pandas
pdf = df_departamento_del_hecho_dane.toPandas()

plt.figure(figsize=(10,5))

plt.bar(pdf["departamento_del_hecho_dane"], pdf["cantidad"])

plt.title("Número de casos por departamento donde ocurrió el hecho")
plt.xlabel("Departamento")
plt.ylabel("Cantidad")

plt.xticks(rotation=90)
plt.grid(axis='y', alpha=0.3)

plt.show()

In [0]:
# Gráfica de barras municipio_del_hecho_dane

#Agrupar por categoría, contar casos y ordenar
df_municipio_del_hecho_dane = (
    df_VIF_VP_no_fatales
    .groupBy("municipio_del_hecho_dane")
    .agg(count("*").alias("cantidad"))
    .orderBy(col("cantidad").desc())
    .limit(20)
)


# Convertir a pandas
pdf = df_municipio_del_hecho_dane.toPandas()

plt.figure(figsize=(10,5))

plt.barh(pdf["municipio_del_hecho_dane"], pdf["cantidad"])

plt.title("Número de casos por municipio donde ocurrió el hecho")
plt.xlabel("Cantidad")
plt.ylabel("Municipio")

plt.subplots_adjust(left=0.35)
plt.yticks(fontsize=9)
plt.gca().invert_yaxis()
plt.xticks(rotation=90)
plt.grid(axis='x', alpha=0.3)

plt.show()

In [0]:
#Valores unicos de la variable localidad del hecho
display(
    df_VIF_VP_no_fatales
    .select("localidad_del_hecho")
    .distinct()
    .orderBy("localidad_del_hecho")
)

In [0]:
# Gráfica de barras de localidad_del_hecho

#Se denomina localidad a una unidad administrativa de la ciudad principal del país que agrupa sectores o barrios determinados.

#Agrupar por categoría, contar casos y ordenar
df_localidad_del_hecho = (
    df_VIF_VP_no_fatales
    .groupBy("localidad_del_hecho")
    .agg(count("*").alias("cantidad"))
    .orderBy(col("cantidad").desc())
    .limit(21)
)


# Convertir a pandas
pdf = df_localidad_del_hecho.toPandas()

plt.figure(figsize=(10,5))

plt.barh(pdf["localidad_del_hecho"], pdf["cantidad"])

plt.title("Número de casos por localidad donde ocurrió el hecho")
plt.xlabel("Cantidad")
plt.ylabel("Localidad")

plt.subplots_adjust(left=0.35)
plt.yticks(fontsize=9)
plt.gca().invert_yaxis()
plt.xticks(rotation=90)
plt.grid(axis='x', alpha=0.3)

plt.show()



In [0]:
#Valores unicos de la variable zona_del_hecho
display(
    df_VIF_VP_no_fatales
    .select("zona_del_hecho")
    .distinct()
    .orderBy("zona_del_hecho")
)

In [0]:
# Gráfica de barras zona_del_hecho

#Clasificación del territorio que diferencia los espacios comprendidos dentro del casco urbano de un municipio y los centros #poblados (zona urbana), de los que están fuera de él (zona rural). Explícitamente se refiere a la zona donde se presentaron #los hechos.

#Recodificar valores de la variable zona_del_hecho
df_VIF_VP_no_fatales = df_VIF_VP_no_fatales.withColumn(
    "zona_del_hecho",
    when(col("zona_del_hecho") == "Centro poblado (corregimiento, inspección de policía y caserío)", 
                                  "Centro poblado(corregimiento, inspección de policía y caserío)")     
    .otherwise(col("zona_del_hecho"))
)

#Agrupar por categoría, contar casos y ordenar
df_zona_del_hecho = (
    df_VIF_VP_no_fatales
    .groupBy("zona_del_hecho")
    .agg(count("*").alias("cantidad"))
    .orderBy(col("cantidad").desc())    
)

# Convertir a pandas
pdf = df_zona_del_hecho.toPandas()

plt.figure(figsize=(10,5))

plt.barh(pdf["zona_del_hecho"], pdf["cantidad"])

plt.title("Número de casos por zona del hecho")
plt.xlabel("Cantidad")
plt.ylabel("Zona del hecho")
plt.gca().invert_yaxis()
plt.subplots_adjust(left=0.35)
plt.yticks(fontsize=9)
plt.xticks(rotation=90)
plt.grid(axis='x', alpha=0.3)

plt.show()

In [0]:
#Valores unicos de la variable escenario_del_hecho
display(
    df_VIF_VP_no_fatales
    .select("escenario_del_hecho")
    .distinct()
    .orderBy("escenario_del_hecho")
)

In [0]:
#Gráfica de barras de escenario_del_hecho

#Clasificación del lugar donde ocurrieron los hechos.

#Recodificar valores de la variable escenario_del_hecho
df_VIF_VP_no_fatales = df_VIF_VP_no_fatales.withColumn(
    "escenario_del_hecho",
    when(col("escenario_del_hecho") == "Ambulancia - Transporte Sanitario", 
                                        "Ambulancia - transporte sanitario")
    .when(col("escenario_del_hecho") == "Calle (Autopista,Avenida,Dentro de La Ciudad)",                                       "Calle (autopista, avenida, dentro de la ciudad)")
    .when(col("escenario_del_hecho") == "Carretera (Fuera de La Ciudad)", "Carretera (fuera de la ciudad)")
    .when(col("escenario_del_hecho") == "Centro de Atención Médica (Hospital,Clínica,Consultorio,Etc)", 
                                        "Centro de atención médica (hospital, clínica, consultorio, etc.)")    
    .when(col("escenario_del_hecho") == "Centros Educativos", "Centros educativos")
    .when(col("escenario_del_hecho") == "Centros de Reclusión", "Centros de reclusión")

    .when(col("escenario_del_hecho") == "Espacios Acuáticos al Aire Libre (Mar,Río,Arroyo,Humedal,Lago,Etc)", "Espacios acuáticos al aire libre (mar, rio, arroyo, humedal, lago, etc.)")

    .when(col("escenario_del_hecho") == "Espacios acuáticos al aire libre (mar, río, arroyo, humedal, lago, etc.)", "Espacios acuáticos al aire libre (mar, rio, arroyo, humedal, lago, etc.)")

    .when(col("escenario_del_hecho") == "Espacios Terrestres al Aire Libre (Bosque,Potrero,Montaña,Playa,Etc)", 
                                        "Espacios terrestres al aire libre (bosque, potrero, montaña, playa, etc.)")
    
    .when(col("escenario_del_hecho") == "Establecimiento Comercial (Tienda,Centro Comercial,Almacén,Plaza de Mercado)",    "Establecimiento comercial (tienda, centro comercial, almacén, plaza de mercado)" )     
    
    .when(col("escenario_del_hecho") == "Establecimiento comercial (tienda, centro comercial, almacen)",
                                        "Establecimiento comercial (tienda, centro comercial, almacén, plaza de mercado)" )
    
    .when(col("escenario_del_hecho") == "Establecimiento comercial (plaza de mercado, galería)",
                                        "Establecimiento comercial (tienda, centro comercial, almacén, plaza de mercado)")

    .when(col("escenario_del_hecho") == "Establecimiento Industrial (Fábrica,Planta) y/o Obras en Construcción","Establecimiento industrial (fábrica, planta) y/o obras en construcción")

    .when(col("escenario_del_hecho") == "Establecimientos Dedicados a la Administración Pública (Cortes,Juzgados,Ministerios,Etc)", "Establecimientos dedicados a la administración pública (cortes, juzgados, ministerios, etc.)")
    .when(col("escenario_del_hecho") == "Establecimientos Financieros y Relacionados (Bancos,Fiduciarias,Etc)","Establecimientos financieros y relacionados (bancos, fiduciarias, etc.)")
    .when(col("escenario_del_hecho") == "Establecimientos de Expendio de Comidas (Restaurantes,Asaderos,Salsamentarias,Etc)", "Establecimientos de expendio de comidas (restaurantes, asaderos, salsamentarias, etc.)")
    .when(col("escenario_del_hecho") == "Estaciones de Servicio (Bombas de Gasolina)", "Estaciones de servicio (bombas de gasolina)")
    .when(col("escenario_del_hecho") == "Guarniciones Militares y/o de Policía", "Guarniciones militares y/o de policía")
    .when(col("escenario_del_hecho") == "Lugares de Actividades Culturales (Cines,Teatros,Museos,Bibliotecas,Etc)","Lugares de actividades culturales (cines, teatros, museos, bibliotecas, etc.)")  
    .when(col("escenario_del_hecho") == "Lugares de Cuidado de Personas (Hospicios,Orfelinatos,Hogares Geriatricos,Etc)","Lugares de cuidado de personas (hospicios, orfelinatos, hogares geriátricos, etc.)")

    .when(col("escenario_del_hecho") == "Lugares de Esparcimiento con Expendio de Alcohol","Lugares de esparcimiento con expendio de alcohol")

    .when(col("escenario_del_hecho") == "Lugares de Hospedaje (Hoteles,Campamentos Y Otros Tipos de Hospedaje No Permanente,Moteles,Etc)", "Lugares de hospedaje (hoteles, campamentos y otros tipos de hospedaje no permanente, moteles, etc.)")

    .when(col("escenario_del_hecho") == "Oficinas y/o Edificios de Oficinas", "Oficinas y/o edificios de oficinas")
    
    .when(col("escenario_del_hecho") == "Parqueaderos,Estacionamientos","Parqueaderos, estacionamientos")
    
    .when(col("escenario_del_hecho") == "Sin Información", "Sin información")

    .when(col("escenario_del_hecho") == "Sitio de Culto (Capilla,Iglesia,Templo,Etc)", "Sitio de culto (capilla, iglesia, templo, etc.)")

    .when(col("escenario_del_hecho") == "Terminales de Pasajeros", "Terminales de pasajeros" )

    .when(col("escenario_del_hecho") == "Terreno Baldío", "Terreno baldío" )

    .when(col("escenario_del_hecho") == "Vehículo Servicio Particular","Vehículo de servicio particular" )

    .when(col("escenario_del_hecho") == "Vía Pública", "Vía pública")

    .when(col("escenario_del_hecho") == "Zonas de Actividades Agropecuarias","Zonas de actividades agropecuarias" )

    .when(col("escenario_del_hecho") == "Áreas Deportivas y/o Recreativas", "Áreas deportivas y/o recreativas")
       
    .otherwise(col("escenario_del_hecho"))
)

#Agrupar por categoría, contar casos y ordenar
df_escenario_del_hecho = (
    df_VIF_VP_no_fatales
    .groupBy("escenario_del_hecho")
    .agg(count("*").alias("cantidad"))
    .orderBy(col("cantidad").desc())
)

# Convertir a pandas
pdf = df_escenario_del_hecho.toPandas()

plt.figure(figsize=(10,5))

plt.barh(pdf["escenario_del_hecho"], pdf["cantidad"])

plt.title("Número de casos por escenario del hecho")
plt.xlabel("Cantidad")
plt.ylabel("Escenario")

plt.subplots_adjust(left=0.35)
plt.yticks(fontsize=8)
plt.gca().invert_yaxis()
plt.xticks(rotation=90)
plt.grid(axis='x', alpha=0.3)

plt.show()



In [0]:
#Valores unicos de la variable escenario_del_hecho
display(
    df_VIF_VP_no_fatales
    .select("escenario_del_hecho")
    .distinct()
    .orderBy("escenario_del_hecho")
)

In [0]:
#Valores unicos de la variable actividad_durante_hecho
display(
    df_VIF_VP_no_fatales
    .select("actividad_durante_hecho")
    .distinct()
    .orderBy("actividad_durante_hecho")
)

In [0]:
#actividad_durante_hecho

#Clasificación de las tareas u operaciones que se encontraba realizando la persona al momento de la lesión.

#Recodificar valores de la variable actividad_durante_hecho
df_VIF_VP_no_fatales = df_VIF_VP_no_fatales.withColumn(
    "actividad_durante_hecho",
    when(col("actividad_durante_hecho") == "Actividades de desplazamiento de un lugar a otro.", 
                                        "Actividades de desplazamiento de un lugar a otro")
    .when(col("actividad_durante_hecho") == "Actividades relacionadas con el aprendizaje",
                                        "Actividades relacionadas con el estudio y el aprendizaje")
    .when(col("actividad_durante_hecho") == "Actividades relacionadas con los deportes y el ejercicio físico.",
                                         "Actividades relacionadas con los deportes y el ejercicio físico")
    .when(col("actividad_durante_hecho") == "Actividades relacionadas con manifestaciones públicas (Marchas, protestas, etc)", "Actividades relacionadas con manifestaciones públicas (marchas, protestas, etc.)") 
    .when(col("actividad_durante_hecho") == "Actividades relacionadas con manifestaciones públicas (Marchas, protestas, etc.)","Actividades relacionadas con manifestaciones públicas (marchas, protestas, etc.)")    
    .otherwise(col("actividad_durante_hecho"))
)

#Agrupar por categoría, contar casos y ordenar
df_actividad_durante_hecho = (
    df_VIF_VP_no_fatales
    .groupBy("actividad_durante_hecho")
    .agg(count("*").alias("cantidad"))
    .orderBy(col("cantidad").desc())    
)

# Convertir a pandas
pdf = df_actividad_durante_hecho.toPandas()

plt.figure(figsize=(10,5))

plt.barh(pdf["actividad_durante_hecho"], pdf["cantidad"])

plt.title("Número de casos por actividad durante el hecho")
plt.xlabel("Cantidad")
plt.ylabel("Actividad")

plt.subplots_adjust(left=0.35)
plt.yticks(fontsize=9)
plt.gca().invert_yaxis()
plt.xticks(rotation=90)
plt.grid(axis='x', alpha=0.3)

plt.show()

In [0]:
#Valores unicos de la variable circunstancia_del_hecho_detallada
display(
    df_VIF_VP_no_fatales
    .select("circunstancia_del_hecho_detallada")
    .distinct()
    .orderBy("circunstancia_del_hecho_detallada")
)

In [0]:
#circunstancia_del_hecho_detallada

#Establece una relación entre la víctima y la situación del entorno en el momento de los hechos. Dicha relación con el entorno #está determinada fundamentalmente por la posibilidad de reconocer la presencia o ausencia de intervención humana

#Agrupar por categoría, contar casos y ordenar
df_circunstancia_del_hecho_detallada = (
    df_VIF_VP_no_fatales
    .groupBy("circunstancia_del_hecho_detallada")
    .agg(count("*").alias("cantidad"))
    .orderBy(col("cantidad").desc())    
)

# Convertir a pandas
pdf = df_circunstancia_del_hecho_detallada.toPandas()

plt.figure(figsize=(10,5))

plt.barh(pdf["circunstancia_del_hecho_detallada"], pdf["cantidad"])

plt.title("Número de casos por circunstancia del hecho")
plt.xlabel("Cantidad")
plt.ylabel("Circunstancia del hecho")

plt.subplots_adjust(left=0.35)
plt.yticks(fontsize=9)
plt.gca().invert_yaxis()
plt.xticks(rotation=90)
plt.grid(axis='x', alpha=0.3)

plt.show()

In [0]:
#Valores unicos de la variable circunstancia_del_hecho_detallada
display(
    df_VIF_VP_no_fatales
    .select("contexto_del_hecho")
    .distinct()
    .orderBy("contexto_del_hecho")
)

In [0]:
#contexto_del_hecho

#Se refiere al contexto de violencia no fatal en la cual se produce la lesión o agresión (lesiones por violencia #interpersonal, presunto delito sexual, lesiones por violencia intrafamiliar, lesiones por eventos de transporte y lesiones #accidentales).

#Recodificar valores de la variable contexto_del_hecho
df_VIF_VP_no_fatales = df_VIF_VP_no_fatales.withColumn(
    "contexto_del_hecho",
    when(col("contexto_del_hecho") == "3 Lesiones no Fatales contra Niños, Niñas y Adolescentes por Violencia Intrafamiliar","Lesiones no Fatales contra Niños, Niñas y Adolescentes por Violencia Intrafamiliar")
    .when(col("contexto_del_hecho") == "4 Lesiones no Fatales por Violencia entre otros Familiares",
                                      "Lesiones no Fatales por Violencia entre otros Familiares")
    .when(col("contexto_del_hecho") =="5 Lesiones no Fatales contra el Adulto Mayor por Violencia Intrafamiliar", "Lesiones no Fatales contra el Adulto Mayor por Violencia Intrafamiliar")
    .when(col("contexto_del_hecho") =="6 Lesiones no Fatales por Violencia de Pareja", "Lesiones no Fatales por Violencia de Pareja")
    .otherwise(col("contexto_del_hecho"))
)

#Agrupar por categoría, contar casos y ordenar
df_contexto_del_hecho = (
    df_VIF_VP_no_fatales
    .groupBy("contexto_del_hecho")
    .agg(count("*").alias("cantidad"))
    .orderBy(col("cantidad").desc())    
)

# Convertir a pandas
pdf = df_contexto_del_hecho.toPandas()

plt.figure(figsize=(10,5))

plt.barh(pdf["contexto_del_hecho"], pdf["cantidad"])

plt.title("Número de casos por contexto del hecho")
plt.xlabel("Cantidad")
plt.ylabel("Contexto del hecho")

plt.subplots_adjust(left=0.35)
plt.yticks(fontsize=9)
plt.gca().invert_yaxis()
plt.xticks(rotation=90)
plt.grid(axis='x', alpha=0.3)

plt.show()



In [0]:
#Valores unicos de la variable mecanismo_causal 
display(
    df_VIF_VP_no_fatales
    .select("mecanismo_causal")
    .distinct()
    .orderBy("mecanismo_causal")
)

In [0]:
#mecanismo_causal

#Corresponde a la clasificación de los tipos de mecanismos fisiopatológicos que conllevaron a la lesión de la persona. Se #clasifican los casos de lesiones de acuerdo a la causa principal que la originó, pueden ser por algún tipo de trauma o lesión #externa. 

#Agrupar por categoría, contar casos y ordenar
df_mecanismo_causal = (
    df_VIF_VP_no_fatales
    .groupBy("mecanismo_causal")
    .agg(count("*").alias("cantidad"))
    .orderBy(col("cantidad").desc())    
)

# Convertir a pandas
pdf = df_mecanismo_causal.toPandas()

plt.figure(figsize=(10,5))

plt.barh(pdf["mecanismo_causal"], pdf["cantidad"])

plt.title("Número de casos por mecanismo causal")
plt.xlabel("Cantidad")
plt.ylabel("Mecanismo causal")

plt.subplots_adjust(left=0.35)
plt.yticks(fontsize=9)
plt.gca().invert_yaxis()
plt.xticks(rotation=90)
plt.grid(axis='x', alpha=0.3)

plt.show()





In [0]:
#Valores unicos de la diagnostico_topográfico
display(
    df_VIF_VP_no_fatales
    .select("diagnostico_topografico")
    .distinct()
    .orderBy("diagnostico_topografico")
)

In [0]:
#diagnostico_topográfico

#Zona anatómica general establecida por la valoración médico legal, donde se produjo la lesión.

#Agrupar por categoría, contar casos y ordenar
df_diagnostico_topografico = (
    df_VIF_VP_no_fatales
    .groupBy("diagnostico_topografico")
    .agg(count("*").alias("cantidad"))
    .orderBy(col("cantidad").desc())    
)

# Convertir a pandas
pdf = df_diagnostico_topografico.toPandas()

plt.figure(figsize=(10,5))

plt.barh(pdf["diagnostico_topografico"], pdf["cantidad"])

plt.title("Número de casos por diagnóstico topográfico")
plt.xlabel("Cantidad")
plt.ylabel("Diagnóstico topográfico")

plt.subplots_adjust(left=0.35)
plt.yticks(fontsize=9)
plt.gca().invert_yaxis()
plt.xticks(rotation=90)
plt.grid(axis='x', alpha=0.3)

plt.show()

In [0]:
#Valores unicos sexo_del_agresor
display(
    df_VIF_VP_no_fatales
    .select("sexo_del_agresor")
    .distinct()
    .orderBy("sexo_del_agresor")
)

In [0]:
#sexo_del_agresor

#Se refiere a la variable biológica que clasifica a la población en hombres y mujeres.En este caso específico hace referencia al sexo del agresor, según el relato de la víctima.

#Agrupar por categoría, contar casos y ordenar
df_sexo_del_agresor = (
    df_VIF_VP_no_fatales
    .groupBy("sexo_del_agresor")
    .agg(count("*").alias("cantidad"))
    .orderBy(col("cantidad").desc())    
)

# Convertir a pandas
pdf = df_sexo_del_agresor.toPandas()

plt.figure(figsize=(10,5))

plt.barh(pdf["sexo_del_agresor"], pdf["cantidad"])

plt.title("Número de casos por sexo del agresor")
plt.xlabel("Cantidad")
plt.ylabel("Sexo del agresor")

plt.subplots_adjust(left=0.35)
plt.yticks(fontsize=9)
plt.gca().invert_yaxis()
plt.xticks(rotation=90)
plt.grid(axis='x', alpha=0.3)

plt.show()


In [0]:
#Valores unicos presunto_agresor
display(
    df_VIF_VP_no_fatales
    .select("presunto_agresor")
    .distinct()
    .orderBy("presunto_agresor")
)

In [0]:
#presunto_agresor

#Caracterización de la persona que se presume, o se sabe, ha sido el causante de la lesión. 

#Recodificar valores de la variable presunto_agresor
df_VIF_VP_no_fatales = df_VIF_VP_no_fatales.withColumn(
    "presunto_agresor",
    when(col("presunto_agresor") == "Abuelo (a)","Abuelo(a)")
    .when(col("presunto_agresor") == "Cuñado (a)","Cuñado(a)")
    .when(col("presunto_agresor") == "Hermano (a)","Hermano(a)")
    .when(col("presunto_agresor") == "Hijo (a)","Hijo(a)")
    .when(col("presunto_agresor") == "NUERA","Nuera")
    .when(col("presunto_agresor") == "Nieto (a)","Nieto(a)")
    .when(col("presunto_agresor") == "Otros familiares civiles o consanguineos","Otros familiares civiles o consanguíneos")
    .when(col("presunto_agresor") == "Primo (a)","Primo(a)")
    .when(col("presunto_agresor") == "Sobrino (a)","Sobrino(a)") 
    .when(col("presunto_agresor") == "Suegro (a)", "Suegro(a)") 
    .when(col("presunto_agresor") == "Tio (a)","Tío(a)")
    .when(col("presunto_agresor") == "Tío (a)","Tío(a)")
    .when(col("presunto_agresor") == "YERNO","Yerno")
    .otherwise(col("presunto_agresor"))
)

#Agrupar por categoría, contar casos y ordenar
df_presunto_agresor = (
    df_VIF_VP_no_fatales
    .groupBy("presunto_agresor")
    .agg(count("*").alias("cantidad"))
    .orderBy(col("cantidad").desc())    
)

# Convertir a pandas
pdf = df_presunto_agresor.toPandas()

plt.figure(figsize=(10,5))

plt.barh(pdf["presunto_agresor"], pdf["cantidad"])

plt.title("Número de casos por presunto agresor")
plt.xlabel("Cantidad")
plt.ylabel("Presunto agresor")

plt.subplots_adjust(left=0.35)
plt.yticks(fontsize=9)
plt.gca().invert_yaxis()
plt.xticks(rotation=90)
plt.grid(axis='x', alpha=0.3)

plt.show()




In [0]:
#Valores unicos factor_desencadenante_agresion
display(
    df_VIF_VP_no_fatales
    .select("factor_desencadenante_agresion")
    .distinct()
    .orderBy("factor_desencadenante_agresion")
)

In [0]:
#factor_desencadenante_agresion

#Factor de riesgo y/o causa circunstancial, el cual desencadenó la agresión hacia la víctima, referido por la víctima. 

#Agrupar por categoría, contar casos y ordenar
df_factor_desencadenante_agresion = (
    df_VIF_VP_no_fatales
    .groupBy("factor_desencadenante_agresion")
    .agg(count("*").alias("cantidad"))
    .orderBy(col("cantidad").desc())    
)

# Convertir a pandas
pdf = df_factor_desencadenante_agresion.toPandas()

plt.figure(figsize=(10,5))

plt.barh(pdf["factor_desencadenante_agresion"], pdf["cantidad"])

plt.title("Número de casos factor desencadenante de la agresion")
plt.xlabel("Cantidad")
plt.ylabel("factor desencadenante de la agresion")

plt.subplots_adjust(left=0.35)
plt.yticks(fontsize=9)
plt.gca().invert_yaxis()
plt.xticks(rotation=90)
plt.grid(axis='x', alpha=0.3)

plt.show()





In [0]:
#Valores unicos dias_de_incapacidad_medicolegal 
display(
    df_VIF_VP_no_fatales
    .select("dias_de_incapacidad_medicolegal")
    .distinct()
    .orderBy("dias_de_incapacidad_medicolegal")
)

In [0]:
#dias_de_incapacidad_medicolegal

#Es el parámetro forense en Colombia basado en el tiempo en días que toma la reparación de las lesiones, en el marco de un #proceso judicial por lesiones no fatales. 

#Recodificar valores de la variable presunto_agresor
df_VIF_VP_no_fatales = df_VIF_VP_no_fatales.withColumn(
    "dias_de_incapacidad_medicolegal",
    when(col("dias_de_incapacidad_medicolegal") == 'Cero', '0 días')
    .when(col("dias_de_incapacidad_medicolegal") == 'Cero días', '0 días')
    .when(col("dias_de_incapacidad_medicolegal") == 'Cero días y Sin información','0 días')
    .when(col("dias_de_incapacidad_medicolegal") == 'Cero días y Sin información','0 días')
    .when(col("dias_de_incapacidad_medicolegal") == '1 a 30', '1 a 30 días')
    .when(col("dias_de_incapacidad_medicolegal") == '31 a 90','31 a 90 días')
    .when(col("dias_de_incapacidad_medicolegal") == 'Más de 90', 'Más de 90 días')
    .otherwise(col("dias_de_incapacidad_medicolegal"))
)

#Agrupar por categoría, contar casos y ordenar
df_dias_de_incapacidad_medicolegal = (
    df_VIF_VP_no_fatales
    .groupBy("dias_de_incapacidad_medicolegal")
    .agg(count("*").alias("cantidad"))
    .orderBy(col("cantidad").desc())    
)

# Convertir a pandas
pdf = df_dias_de_incapacidad_medicolegal.toPandas()

plt.figure(figsize=(10,5))

plt.barh(pdf["dias_de_incapacidad_medicolegal"], pdf["cantidad"])

plt.title("Número de casos por dias de incapacidad_medicolegal")
plt.xlabel("Cantidad")
plt.ylabel("Días")

plt.subplots_adjust(left=0.35)
plt.yticks(fontsize=9)
plt.gca().invert_yaxis()
plt.xticks(rotation=90)
plt.grid(axis='x', alpha=0.3)

plt.show()

In [0]:
#Valores unicos pueblo_indigena 
display(
    df_VIF_VP_no_fatales
    .select("pueblo_indigena")
    .distinct()
    .orderBy("pueblo_indigena")
)

In [0]:
#pueblo_indigena

#Agrupar por categoría, contar casos y ordenar
df_pueblo_indigena = (
    df_VIF_VP_no_fatales
    .groupBy("pueblo_indigena")
    .agg(count("*").alias("cantidad"))
    .orderBy(col("cantidad").desc())
    .limit(20)    
)

# Convertir a pandas
pdf = df_pueblo_indigena.toPandas()

plt.figure(figsize=(10,5))

plt.barh(pdf["pueblo_indigena"], pdf["cantidad"])

plt.title("Número de casos segun pueblo_indigena")
plt.xlabel("Cantidad")
plt.ylabel("Pueblo indígena")

plt.subplots_adjust(left=0.35)
plt.yticks(fontsize=9)
plt.gca().invert_yaxis()
plt.xticks(rotation=90)
plt.grid(axis='x', alpha=0.3)

plt.show()

Guardar la tabla transformada

In [0]:
#guardar la tabla transformada en delta

df_VIF_VP_no_fatales.write \
    .mode("overwrite") \
    .saveAsTable(
        "ml_proyecto_7405607705157039.default.VIF_VP_no_fatales_v1"
    )